# Arabizi keyboard: training sweep for Llama 3.2 1B (accuracy plan, step 2)

Four LoRA runs on the **same export** as the shipped model, each changing **one** setting (docs/TRAINING.md, "Sweep"):

| Run | Change from the shipped settings (rank 16, alpha 32, lr 2e-4, 3 epochs) |
|---|---|
| `base` | none: checks that a fresh session reproduces the shipped model (P4 gate) and shows how much a rerun alone moves the score |
| `rank32` | LoRA rank 32, alpha 64 |
| `lr1e-4` | learning rate 1e-4 |
| `ep2` | 2 epochs |

**How to run:** Settings → Accelerator → **GPU T4** · **Internet on** · Add Input → your private `arabizi-train` dataset. Then **Save Version → Save & Run All (Commit)**: it runs in the background (about 2 hours) and keeps the outputs on the version page, even if you close the tab.

**Privacy:** the dataset holds Moustafa's own messages (ADR 0014). Keep this notebook and its dataset **private**. Dev and test never leave the Mac: the runs are scored there.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU: the same effective batch (32) as the shipped model


In [ ]:
# Fresh copy of the code on every run (an old clone would otherwise be kept and run).
!rm -rf /kaggle/working/repo
!git clone --quiet --depth 1 https://github.com/Mustafa-Abbara/arabizi-keyboard /kaggle/working/repo
!git -C /kaggle/working/repo log -1 --format='repo commit: %h %s'
!pip install --quiet -r /kaggle/working/repo/train/requirements.txt


In [ ]:
import glob
import json
import subprocess
import sys

exports = glob.glob("/kaggle/input/**/export.json", recursive=True)
assert len(exports) == 1, f"add exactly one dataset with export.json (found {exports})"
DATA = os.path.dirname(exports[0])
with open(exports[0]) as f:
    summary = json.load(f)
print(DATA, "|", summary["rows_total"], "examples | AI share", summary["ai_share"], "| prompt v", summary["prompt_version"])
OUT = "/kaggle/working/out"
REPO = "/kaggle/working/repo"


In [ ]:
# One change per run against the shipped settings; the shipped model's run is train/runs/20261016-051159-llama3.2-1b-lora.json.
RUNS = {
    "base": [],
    "rank32": ["--rank", "32", "--alpha", "64"],
    "lr1e-4": ["--lr", "1e-4"],
    "ep2": ["--epochs", "2"],
}

for tag, extra in RUNS.items():
    print(f"\n=== {tag}: {' '.join(extra) or 'shipped settings'}", flush=True)
    # check=True: a failed run stops the notebook instead of exporting nothing.
    subprocess.run([sys.executable, f"{REPO}/train/finetune_lora.py", "--base", "llama3.2-1b", "--data", DATA,
                    "--out", OUT, "--tag", tag, *extra], check=True)
    run = max(glob.glob(f"{OUT}/*-llama3.2-1b-lora-{tag}"))
    subprocess.run(["bash", f"{REPO}/train/export_gguf.sh", f"{run}/merged", f"arabizi-llama3.2-1b-{tag}", run],
                   check=True)
    subprocess.run(["rm", "-rf", f"{run}/merged", f"{run}/checkpoints"], check=True)


## Results and download

The table shows each run's best validation loss (training-style rows: **not** the decision; dev accuracy on the Mac decides). Download from the version's **Output** tab, per run: `arabizi-llama3.2-1b-<tag>-Q4_K_M.gguf`, its `.sha256` and `run.json`. On the Mac: the `.gguf` files go to `models/`, the `run.json` files to `train/runs/<run folder name>.json`; then tell Claude.


In [ ]:
for path in sorted(glob.glob(f"{OUT}/*-lora-*/run.json")):
    with open(path) as f:
        r = json.load(f)
    print(f"{r['run_id']:45} best validation loss {r['best_eval_loss']:.4f}  ({r['minutes']} min)")
!find "$OUT" -maxdepth 2 -type f -printf "%10s  %p\n"
